In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;

Init();

In [ ]:
string proj = "SlipConvergence_Droplet";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
//BoSSSshell.WorkflowMgm.Sessions.ElementAt(5).GetSessionDirectory()

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination);
sessions.ForEach(s => display(s));

In [ ]:
public static Dictionary<string, Plot2Ddata> ExtractConvergenceH(IEnumerable<ISessionInfo> filteredsessions){

    Dictionary<string, Plot2Ddata> ret = new Dictionary<string, Plot2Ddata>();
    
    // ITimestepInfo[] timesteps = filteredsessions.Select(s => s.Timesteps.Last()).ToArray();
    // string[] fields = timesteps.First().Fields.Select(f => f.Identification).ToArray();
    string[] fields = new string[] {"Pressure", "VelocityX", "VelocityY", "Temperature"};
    
    foreach(var field in fields){
        Plot2Ddata errDat = filteredsessions.ToEstimatedGridConvergenceData(field, normType: NormType.L2_embedded); 
        ret.Add(field, errDat);
    } 
    
    return ret;
}

public static Dictionary<string, List<ISessionInfo>> ExtractSessionsByName(IEnumerable<ISessionInfo> sessions){
    
    Dictionary<string, List<ISessionInfo>> filteredsessions = new Dictionary<string, List<ISessionInfo>>();
    
    foreach(var sess in sessions){
        var key = Regex.Split(sess.Name , ".*meshStudy_[0-9]+_P[0-9]?_",RegexOptions.IgnoreCase).Last();
        
        if(!filteredsessions.ContainsKey(key)){
            filteredsessions[key] = new List<ISessionInfo>();
        }
        filteredsessions[key].Add(sess);
    }
    
    return filteredsessions;
}

In [ ]:
using System.Data;

In [ ]:
DataTable table = new DataTable("ConvergenceData");
DataColumn column;
DataRow row;

column = new DataColumn();
column.DataType = System.Type.GetType("System.String");
column.ColumnName = "Study";
column.ReadOnly = true;
column.Unique = true;
table.Columns.Add(column);

column = new DataColumn();
column.DataType = typeof(List<ISessionInfo>);
column.ColumnName = "Sessions";
column.ReadOnly = false;
column.Unique = false;
table.Columns.Add(column);

table.Columns.Add("Timesteps", typeof(List<ITimestepInfo>));
table.Columns.Add("SessionID", typeof(List<Guid>));

table.Columns.Add("P-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("P-PlotData", typeof(Plot2Ddata));

table.Columns.Add("U-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("U-PlotData", typeof(Plot2Ddata));

table.Columns.Add("V-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("V-PlotData", typeof(Plot2Ddata));

table.Columns.Add("Velocity-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("Velocity-PlotData", typeof(Plot2Ddata));

table.Columns.Add("T-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("T-PlotData", typeof(Plot2Ddata));

var filteredsessions = ExtractSessionsByName(sessions);

foreach(var kvp in filteredsessions){
    row = table.NewRow();
    row["Study"] = kvp.Key;
    row["Sessions"] = kvp.Value;
    table.Rows.Add(row);
}

In [ ]:
foreach(DataRow row in table.Rows){
    List<ITimestepInfo> ts = new List<ITimestepInfo>();
    List<Guid> ids = new List<Guid>();

    foreach(ISessionInfo s in (List<ISessionInfo>)row["Sessions"]){
        ts.Add(s.Timesteps.Last());
        ids.Add(s.ID);
    }

    row["Timesteps"] = ts;
    row["SessionID"] = ids;

    row["P-PlotData"] = ts.ToEstimatedGridConvergenceData("Pressure", normType: NormType.L2_embedded);
    row["P-Convergence"] = ((Plot2Ddata)row["P-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

    row["U-PlotData"] = ts.ToEstimatedGridConvergenceData("VelocityX", normType: NormType.L2_embedded);
    row["U-Convergence"] = ((Plot2Ddata)row["U-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

    row["V-PlotData"] = ts.ToEstimatedGridConvergenceData("VelocityY", normType: NormType.L2_embedded);
    row["V-Convergence"] = ((Plot2Ddata)row["V-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

    row["T-PlotData"] = ts.ToEstimatedGridConvergenceData("Temperature", normType: NormType.L2_embedded);
    row["T-Convergence"] = ((Plot2Ddata)row["T-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

    Console.WriteLine("Done with: " + row["Study"]);
}

In [ ]:
table.Print()

In [ ]:
foreach(DataRow row in table.Rows){
    var ux = (Plot2Ddata)row["U-PlotData"] ;
    var uy = (Plot2Ddata)row["V-PlotData"] ;
    
    Dictionary<string, double[][]> dataGroups = new Dictionary<string, double[][]>();    
    foreach(var groupX in ux.dataGroups){
        var groupY = uy.dataGroups.Where(g => g.Name == groupX.Name).Single();
        double[] xValues = groupX.Abscissas;
        double[] yValues = groupX.Values.Zip(groupY.Values, (a, b) => Math.Sqrt(a*a+b*b)).ToArray();
        dataGroups.Add(groupX.Name, new double[2][] { xValues, yValues });
    }
    
    row["Velocity-PlotData"] = new Plot2Ddata(dataGroups.ToArray()).WithLogX().WithLogY();
    row["Velocity-Convergence"] = ((Plot2Ddata)row["Velocity-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

}

Plot export routine (activate the corresponding line if necessary)

In [ ]:
foreach(DataRow study2export in table.Rows){
    var study = (string)study2export["Study"];
    var studysessions = (List<ISessionInfo>)study2export["Sessions"];  
    var projectpath = Path.GetFullPath(Path.Join(".", BoSSSshell.wmg.CurrentProject)); 
    var studypath = Path.GetFullPath(Path.Join(projectpath, study));    
    

    foreach(var studysession in studysessions){
        var name = studysession.Name;
        var degree = Regex.Match(name, "P[0-9]{1}").Value;
        var mesh = Regex.Match(name, "meshStudy_[0-9]+").Value;    

        if(!Directory.Exists(Path.Join(studypath, degree))){
            Directory.CreateDirectory(Path.Join(studypath, degree));
        }  

        var fields = studysession.Timesteps.Last().Fields;

        List<DGField> Fields2Plot = new List<DGField>();
        
        foreach (var field in fields) {
            if (field is XDGField xField) {
                foreach (var spc in xField.Basis.Tracker.SpeciesNames) {
                    Fields2Plot.Add(xField.GetSpeciesShadowField(spc));
                }
            } else {
                Fields2Plot.Add(field);
            }
        }
        
        //BoSSS.Solution.Tecplot.Tecplot.PlotFields(Fields2Plot, Path.Join(studypath, degree, mesh) + ".plt", 0.0, 2); // uncomment to export

    }
}

In [ ]:
static IEnumerable<DataRow> ExtractSessionsForStage(this DataTable table, int stage){

    switch(stage){
        case 1:{
        return table.AsEnumerable().Where(row => row.Field<string>("Study").Contains("ST0.10_HV-Infinity"));
        }
        case 2:{
        return table.AsEnumerable().Where(row => row.Field<string>("Study").Contains("ST0.10_HV1,000.00"));
        }
        case 3:{
        return table.AsEnumerable().Where(row => row.Field<string>("Study").Contains("ST0.00_HV1,000.00"));
        }
        default:{
            throw new NotImplementedException();
        }
    }
}

static void VizualizeStage(this DataTable table, int stage){
    Console.WriteLine("".PadRight(100,'/'));
    Console.WriteLine("".PadRight(100,'\\'));
    Console.WriteLine("Results for stage {0}", stage);
    
    var rows = table.ExtractSessionsForStage(stage);
    
    foreach(DataRow row in rows){
        Console.WriteLine("".PadRight(100,'/'));
        Console.WriteLine("".PadRight(100,'\\'));
        Console.WriteLine(row["Study"]);

        foreach(var kvp in ((Dictionary<int, double>)row["P-Convergence"])){
            Console.WriteLine($"{kvp.Key} : {kvp.Value}");
        }
        ((Plot2Ddata)row["P-PlotData"]).ModFormat();
        ((Plot2Ddata)row["P-PlotData"]).PlotNow().Display();

        foreach(var kvp in ((Dictionary<int, double>)row["Velocity-Convergence"])){
            Console.WriteLine($"{kvp.Key} : {kvp.Value}");
        }
        ((Plot2Ddata)row["Velocity-PlotData"]).ModFormat();
        ((Plot2Ddata)row["Velocity-PlotData"]).PlotNow().Display();

        foreach(var kvp in ((Dictionary<int, double>)row["T-Convergence"])){
            Console.WriteLine($"{kvp.Key} : {kvp.Value}");
        } 
        ((Plot2Ddata)row["T-PlotData"]).ModFormat();
        ((Plot2Ddata)row["T-PlotData"]).PlotNow().Display();
    }
}

static void WriteTexTableForStage(this DataTable table, int stage){

    var rows = table.ExtractSessionsForStage(stage);
    
    string[] angles = new string[] {"CA90.00", "CA80.00"};
    string[] sliplenghts = new string[] {"SL0.00", "SL0.10", "SLInfinity"};
    
    Dictionary<int, double>[,] ConvergenceRatesP = new Dictionary<int, double>[angles.Length,sliplenghts.Length];
    Dictionary<int, double>[,] ConvergenceRatesU = new Dictionary<int, double>[angles.Length,sliplenghts.Length];
    Dictionary<int, double>[,] ConvergenceRatesT = new Dictionary<int, double>[angles.Length,sliplenghts.Length];

    for(int i = 0; i < angles.Length; i++){
        for(int j = 0; j < sliplenghts.Length; j++){
            var row = rows.Where(r => ((string)r["Study"]).Contains(angles[i]) && ((string)r["Study"]).Contains(sliplenghts[j])).Single();
            ConvergenceRatesP[i,j] = (Dictionary<int, double>)row["P-Convergence"];
            ConvergenceRatesU[i,j] = (Dictionary<int, double>)row["Velocity-Convergence"];
            ConvergenceRatesT[i,j] = (Dictionary<int, double>)row["T-Convergence"];
        }
    }

    
    using(var stw = new StringWriter()) {    
        stw.WriteLine(@"\begin{table}[t!]");
        stw.WriteLine(@"    \centering");
        stw.WriteLine(@"    \caption{Stage"+$"{stage}"+@"}");
        stw.WriteLine(@"    \label{tab:Stage"+$"{stage}"+@"}");
        stw.WriteLine(@"    \begin{tabularx}{\linewidth}{ >{\hsize=1.75\hsize \raggedleft\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X  >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X }		");
        stw.WriteLine(@"     & \multicolumn{3}{ c }{$\cAngleStat = 90^\circ $} & \multicolumn{3}{c}{$\cAngleStat = 80^\circ $} \\");
        stw.WriteLine(@"    \cmidrule(lr){2-4}\cmidrule(lr){5-7}");
        stw.WriteLine(@"     $\Pdeg$ & $\iSlip = 0$ & $\iSlip = 0.1$ & $\iSlip = \infty$ & $\iSlip = 0$ & $\iSlip = 0.1$ & $\iSlip = \infty$ \\");
        stw.WriteLine(@"    \rowcolor{gray!20} & & & & & &\\[-1em]");
        stw.WriteLine(@"    \rowcolor{gray!20} \multicolumn{1}{ l }{$\LtwoNorm{\press-\press_{ref}}$} & & & & & & \\");
        for(int i = 1; i<=3; i++){
        stw.WriteLine(@"    \rowcolor{gray!20}"+$@" {i} & {ConvergenceRatesP[0,0][i].ToString("N2")} & {ConvergenceRatesP[0,1][i].ToString("N2")} & {ConvergenceRatesP[0,2][i].ToString("N2")} & {ConvergenceRatesP[1,0][i].ToString("N2")} & {ConvergenceRatesP[1,1][i].ToString("N2")} & {ConvergenceRatesP[1,2][i].ToString("N2")}\\");
        }
        stw.WriteLine(@"    & & & & & &\\[-1em]");
        stw.WriteLine(@"    \multicolumn{1}{ l }{$\LtwoNorm{\veloc-\veloc_{ref}}$} & & & & & & \\");
        for(int i = 2; i<=4; i++){
        stw.WriteLine(@"    "+$@" {i} & {ConvergenceRatesU[0,0][i].ToString("N2")} & {ConvergenceRatesU[0,1][i].ToString("N2")} & {ConvergenceRatesU[0,2][i].ToString("N2")} & {ConvergenceRatesU[1,0][i].ToString("N2")} & {ConvergenceRatesU[1,1][i].ToString("N2")} & {ConvergenceRatesU[1,2][i].ToString("N2")}\\");
        }
        stw.WriteLine(@"    \rowcolor{gray!20}& & & & & &\\[-1em]");
        stw.WriteLine(@"    \rowcolor{gray!20} \multicolumn{1}{ l }{$\LtwoNorm{\temp-\temp_{ref}}$} & & & & & & \\");
        for(int i = 2; i<=4; i++){
        stw.WriteLine(@"    \rowcolor{gray!20}"+$@" {i} & {ConvergenceRatesT[0,0][i].ToString("N2")} & {ConvergenceRatesT[0,1][i].ToString("N2")} & {ConvergenceRatesT[0,2][i].ToString("N2")} & {ConvergenceRatesT[1,0][i].ToString("N2")} & {ConvergenceRatesT[1,1][i].ToString("N2")} & {ConvergenceRatesT[1,2][i].ToString("N2")}\\");
        }
        stw.WriteLine(@"    \end{tabularx}");
        stw.WriteLine(@"\end{table}");
        
        File.WriteAllText($"{BoSSSshell.wmg.CurrentProject}/Stage{stage}Table.txt", stw.ToString());
    }
}

In [ ]:
//table.WriteTexTableForStage(1);
//table.WriteTexTableForStage(2);
//table.WriteTexTableForStage(3);

In [ ]:
var stage1 = table.ExtractSessionsForStage(1);
var stage2 = table.ExtractSessionsForStage(2);
var stage3 = table.ExtractSessionsForStage(3);

In [ ]:
table.VizualizeStage(1);

In [ ]:
table.VizualizeStage(2);

In [ ]:
table.VizualizeStage(3);

In [ ]:
static void WriteVelocityTable(IEnumerable<ISessionInfo> Sessions){
    using(var stw = new StringWriter()) {    
        stw.WriteLine(@"\begin{table}[t!]");
        stw.WriteLine(@"    \centering");
        stw.WriteLine(@"    \caption{Velocities}");
        stw.WriteLine(@"    \label{tab:Velocities}");
        stw.WriteLine(@"    \begin{tabularx}{\linewidth}{ >{\raggedright\arraybackslash}X |>{\centering\arraybackslash}X >{\centering\arraybackslash}X >{\centering\arraybackslash}X} >{\centering\arraybackslash}X");
        stw.WriteLine(@"        Case & $\jump{\veloc\cdot\normalB}$ & $\jump{\veloc\cdot\normalI}$ & $\jump{\veloc\cdot\normalL}$ & \cref{eq:cLconflict} \\");
        stw.WriteLine(@"        \hline");
        foreach(var s in Sessions){
            var path = Path.Combine(s.Database.Path, "sessions", s.ID.ToString(), "SlipDroplet.txt");            
            string[] lines = File.ReadAllLines(path);
            
            var values = lines.Last().Split('\t');
            string UxN = Convert.ToDouble(values[4]).ToString("E3");
            UxN = "$" + UxN.Split('E')[0].PadRight(5, '0').Substring(0, Convert.ToDouble(UxN) > 0 ? 4 : 5) + @"\times 10^{"+ (UxN.Split('E').Length > 1 ? Regex.Replace(UxN.Split('E')[1], "-[0]+", "-") : "0") +"} $";
            string UxI = Convert.ToDouble(values[2]).ToString("E3");
            UxI = "$" + UxI.Split('E')[0].PadRight(5, '0').Substring(0, Convert.ToDouble(UxI) > 0 ? 4 : 5) + @"\times 10^{"+ (UxI.Split('E').Length > 1 ? Regex.Replace(UxI.Split('E')[1], "-[0]+", "-") : "0") +"} $";
            string UxL = Convert.ToDouble(values[3]).ToString("E3");
            UxL = "$" + UxL.Split('E')[0].PadRight(5, '0').Substring(0, Convert.ToDouble(UxL) > 0 ? 4 : 5) + @"\times 10^{"+ (UxL.Split('E').Length > 1 ? Regex.Replace(UxL.Split('E')[1], "-[0]+", "-") : "0") +"} $";
            

                
            double surfT = Convert.ToDouble(s.KeysAndQueries["id:sigma"]);
            double hvap = Convert.ToDouble(s.KeysAndQueries["id:hvap"]);
            double lslip = Convert.ToDouble(s.KeysAndQueries["id:interfacesliplength"]);
            double theta = Convert.ToDouble(s.KeysAndQueries["id:theta"]);
            
            double c = -Math.Cos(theta/90.0*Math.PI/2)*Convert.ToDouble(values[2]) + Math.Sin(theta/90.0*Math.PI/2)*Convert.ToDouble(values[3]);
            string conflict = c.ToString("E3");
            conflict = "$" + conflict.Split('E')[0].PadRight(5, '0').Substring(0, Convert.ToDouble(conflict) > 0 ? 4 : 5) + @"\times 10^{"+ (conflict.Split('E').Length > 1 ? Regex.Replace(conflict.Split('E')[1], "-[0]+", "-") : "0") +"} $";

            stw.WriteLine(@"        & & & & \\[-1em]");
            stw.WriteLine(@"        $\left[ L_{\interface,"+(lslip == double.PositiveInfinity ? @"\infty" : lslip == 0.0 ? "0.0" : "0.1")+@"}, h_{v,"+(hvap == double.NegativeInfinity ? @"\infty" : hvap == 0.0 ? "0.0" : "10^3")+@"}, \surfT_{"+(surfT == 0.0 ? @"0.0" : "0.1")+@"}, \cAngle_{"+(theta == 90.0 ? "90" : "80")+@"^\circ}\right]$&"+UxN+@"&"+UxI+@"&"+UxL+@"&"+conflict+@"\\");
        }
        stw.WriteLine(@"    \end{tabularx}");
        stw.WriteLine(@"\end{table}");
        
        File.WriteAllText($"{BoSSSshell.wmg.CurrentProject}/VelocityTable.txt", stw.ToString());
    }
    
}

In [ ]:
var vsess = sessions.Where(s => Convert.ToInt32(s.KeysAndQueries["id:degree"]) == 4 && Convert.ToInt32(s.KeysAndQueries["id:gridlevel"]) == 128);
vsess = vsess.OrderByDescending(s => Convert.ToDouble(s.KeysAndQueries["id:theta"])).ThenBy(s => Convert.ToDouble(s.KeysAndQueries["id:hvap"])).ThenByDescending(s => Convert.ToDouble(s.KeysAndQueries["id:sigma"])).ThenBy(s => Convert.ToDouble(s.KeysAndQueries["id:interfacesliplength"]));

vsess.Display();

In [ ]:
//WriteVelocityTable(vsess);

Assert slopes to state of thesis $\pm0.1$  
Field order ist $(p, u, T)$

In [ ]:
Dictionary<string, double[]> ControlSlopes = new Dictionary<string, double[]>();

ControlSlopes["P2_CA90.00_SL0.00_ST0.10_HV-Infinity"] = new double[]{2.08, 3.09, 3.21};
ControlSlopes["P3_CA90.00_SL0.00_ST0.10_HV-Infinity"] = new double[]{3.02, 3.96, 3.9};
ControlSlopes["P4_CA90.00_SL0.00_ST0.10_HV-Infinity"] = new double[]{3.84, 4.81, 4.53};
ControlSlopes["P2_CA90.00_SL0.10_ST0.10_HV-Infinity"] = new double[]{2.16, 2.99, 3.21};
ControlSlopes["P3_CA90.00_SL0.10_ST0.10_HV-Infinity"] = new double[]{3.17, 3.83, 3.9};
ControlSlopes["P4_CA90.00_SL0.10_ST0.10_HV-Infinity"] = new double[]{3.78, 4.67, 4.53};
ControlSlopes["P2_CA90.00_SLInfinity_ST0.10_HV-Infinity"] = new double[]{2.34, 3.24, 3.21};
ControlSlopes["P3_CA90.00_SLInfinity_ST0.10_HV-Infinity"] = new double[]{3.45, 4.0, 3.9};
ControlSlopes["P4_CA90.00_SLInfinity_ST0.10_HV-Infinity"] = new double[]{4.03, 4.68, 4.53};
ControlSlopes["P2_CA80.00_SL0.00_ST0.10_HV-Infinity"] = new double[]{1.77, 2.67, 3.25};
ControlSlopes["P3_CA80.00_SL0.00_ST0.10_HV-Infinity"] = new double[]{1.93, 2.95, 3.89};
ControlSlopes["P4_CA80.00_SL0.00_ST0.10_HV-Infinity"] = new double[]{2.29, 2.76, 4.58};
ControlSlopes["P2_CA80.00_SL0.10_ST0.10_HV-Infinity"] = new double[]{1.81, 2.81, 3.25};
ControlSlopes["P3_CA80.00_SL0.10_ST0.10_HV-Infinity"] = new double[]{1.95, 3.04, 3.89};
ControlSlopes["P4_CA80.00_SL0.10_ST0.10_HV-Infinity"] = new double[]{2.24, 2.81, 4.58};
ControlSlopes["P2_CA80.00_SLInfinity_ST0.10_HV-Infinity"] = new double[]{1.63, 2.37, 3.25};
ControlSlopes["P3_CA80.00_SLInfinity_ST0.10_HV-Infinity"] = new double[]{1.61, 2.42, 3.89};
ControlSlopes["P4_CA80.00_SLInfinity_ST0.10_HV-Infinity"] = new double[]{1.84, 2.09, 4.58};

ControlSlopes["P2_CA90.00_SL0.00_ST0.10_HV1,000.00"] = new double[]{1.18, 2.28, 3.21};
ControlSlopes["P3_CA90.00_SL0.00_ST0.10_HV1,000.00"] = new double[]{1.1, 2.51, 3.9};
ControlSlopes["P4_CA90.00_SL0.00_ST0.10_HV1,000.00"] = new double[]{1.17, 2.49, 4.53};
ControlSlopes["P2_CA90.00_SL0.10_ST0.10_HV1,000.00"] = new double[]{1.16, 2.3, 3.21};
ControlSlopes["P3_CA90.00_SL0.10_ST0.10_HV1,000.00"] = new double[]{1.04, 2.53, 3.9};
ControlSlopes["P4_CA90.00_SL0.10_ST0.10_HV1,000.00"] = new double[]{1.05, 2.51, 4.53};
ControlSlopes["P2_CA90.00_SLInfinity_ST0.10_HV1,000.00"] = new double[]{1.05, 2.34, 3.21};
ControlSlopes["P3_CA90.00_SLInfinity_ST0.10_HV1,000.00"] = new double[]{1.03, 2.56, 3.9};
ControlSlopes["P4_CA90.00_SLInfinity_ST0.10_HV1,000.00"] = new double[]{1.05, 2.57, 4.53};
ControlSlopes["P2_CA80.00_SL0.00_ST0.10_HV1,000.00"] = new double[]{0.49, 1.48, 3.25};
ControlSlopes["P3_CA80.00_SL0.00_ST0.10_HV1,000.00"] = new double[]{0.19, 1.23, 3.89};
ControlSlopes["P4_CA80.00_SL0.00_ST0.10_HV1,000.00"] = new double[]{0.07, 1.13, 4.58};
ControlSlopes["P2_CA80.00_SL0.10_ST0.10_HV1,000.00"] = new double[]{0.97, 2.19, 3.25};
ControlSlopes["P3_CA80.00_SL0.10_ST0.10_HV1,000.00"] = new double[]{1.0, 2.08, 3.89};
ControlSlopes["P4_CA80.00_SL0.10_ST0.10_HV1,000.00"] = new double[]{0.78, 1.88, 4.58};
ControlSlopes["P2_CA80.00_SLInfinity_ST0.10_HV1,000.00"] = new double[]{0.93, 2.36, 3.25};
ControlSlopes["P3_CA80.00_SLInfinity_ST0.10_HV1,000.00"] = new double[]{1.1, 2.31, 3.89};
ControlSlopes["P4_CA80.00_SLInfinity_ST0.10_HV1,000.00"] = new double[]{1.01, 2.32, 4.58};

ControlSlopes["P2_CA90.00_SL0.00_ST0.00_HV1,000.00"] = new double[]{1.18, 2.28, 3.21};
ControlSlopes["P3_CA90.00_SL0.00_ST0.00_HV1,000.00"] = new double[]{1.11, 2.51, 3.9};
ControlSlopes["P4_CA90.00_SL0.00_ST0.00_HV1,000.00"] = new double[]{1.2, 2.49, 4.53};
ControlSlopes["P2_CA90.00_SL0.10_ST0.00_HV1,000.00"] = new double[]{1.17, 2.3, 3.21};
ControlSlopes["P3_CA90.00_SL0.10_ST0.00_HV1,000.00"] = new double[]{1.07, 2.53, 3.9};
ControlSlopes["P4_CA90.00_SL0.10_ST0.00_HV1,000.00"] = new double[]{1.14, 2.51, 4.53};
ControlSlopes["P2_CA90.00_SLInfinity_ST0.00_HV1,000.00"] = new double[]{1.07, 2.34, 3.21};
ControlSlopes["P3_CA90.00_SLInfinity_ST0.00_HV1,000.00"] = new double[]{1.07, 2.56, 3.9};
ControlSlopes["P4_CA90.00_SLInfinity_ST0.00_HV1,000.00"] = new double[]{1.13, 2.57, 4.53};
ControlSlopes["P2_CA80.00_SL0.00_ST0.00_HV1,000.00"] = new double[]{0.49, 1.48, 3.25};
ControlSlopes["P3_CA80.00_SL0.00_ST0.00_HV1,000.00"] = new double[]{0.19, 1.23, 3.89};
ControlSlopes["P4_CA80.00_SL0.00_ST0.00_HV1,000.00"] = new double[]{0.07, 1.13, 4.58};
ControlSlopes["P2_CA80.00_SL0.10_ST0.00_HV1,000.00"] = new double[]{0.97, 2.19, 3.25};
ControlSlopes["P3_CA80.00_SL0.10_ST0.00_HV1,000.00"] = new double[]{1.01, 2.08, 3.89};
ControlSlopes["P4_CA80.00_SL0.10_ST0.00_HV1,000.00"] = new double[]{0.85, 1.87, 4.58};
ControlSlopes["P2_CA80.00_SLInfinity_ST0.00_HV1,000.00"] = new double[]{0.94, 2.36, 3.25};
ControlSlopes["P3_CA80.00_SLInfinity_ST0.00_HV1,000.00"] = new double[]{1.12, 2.31, 3.89};
ControlSlopes["P4_CA80.00_SLInfinity_ST0.00_HV1,000.00"] = new double[]{1.08, 2.32, 4.58};

In [ ]:
// todo load row from table and compare slopes
bool success = true;
double thrsh = 0.1;

foreach(var kvp in ControlSlopes){
    double[] cVals = kvp.Value;
    var row = table.AsEnumerable().Where(row => row.Field<string>("Study").Equals(kvp.Key)).Single();
    double[] eVals = new double[]{((Dictionary<int, double>)row["P-Convergence"]).First().Value, ((Dictionary<int, double>)row["Velocity-Convergence"]).First().Value, ((Dictionary<int, double>)row["T-Convergence"]).First().Value};

    for(int i = 0; i < eVals.Length; i++){
        if(eVals[i] < cVals[i] - thrsh || eVals[i] > cVals[i] + thrsh){
            success &= false;
            Console.WriteLine("Mismatch in computed slopes in {0}, for field {1}", kvp.Key, i);
        }
    }
}

if(!success){
    throw new ApplicationException("Not all slopes match!");
}
